# Phân Tích Churn Khách Hàng Trong Lĩnh Vực Viễn Thông

In [ ]:
!pip install pandas
!pip install numpy

In [59]:
import pandas as pd
import numpy as np

In [60]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [61]:
df.info()
#TotalCharges kiểu str

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [62]:
#loại bỏ duplicates dựa trên Customer ID
df_clean = df.drop_duplicates(subset=['customerID'].copy())

#Xử lý missing và ép kiểu TotalCharges
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'],errors='coerce')
df_clean = df_clean.dropna(subset=['TotalCharges']) 

#chuyển đổi SeniorCitizen sang Yes/No
df_clean['SeniorCitizen']= df_clean['SeniorCitizen'].replace({1:'yes', 0:'no'})


df_clean.info()



<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   str    
 1   gender            7032 non-null   str    
 2   SeniorCitizen     7032 non-null   object 
 3   Partner           7032 non-null   str    
 4   Dependents        7032 non-null   str    
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   str    
 7   MultipleLines     7032 non-null   str    
 8   InternetService   7032 non-null   str    
 9   OnlineSecurity    7032 non-null   str    
 10  OnlineBackup      7032 non-null   str    
 11  DeviceProtection  7032 non-null   str    
 12  TechSupport       7032 non-null   str    
 13  StreamingTV       7032 non-null   str    
 14  StreamingMovies   7032 non-null   str    
 15  Contract          7032 non-null   str    
 16  PaperlessBilling  7032 non-null   str    
 17  PaymentMeth

In [63]:
#Tạo cột mới: Tenure Group (nhóm thời gian: <12 tháng, 12-36, >36)
moc_cat = [0, 11.9, 36, float('inf')]
nhan_dan = ['<12 tháng', '12-36 tháng', '>36 tháng']

#Tạo cột Tenure Group
df_clean['Tenure Group']= pd.cut(df_clean['tenure'], bins=moc_cat, labels=nhan_dan)

print(df_clean[['customerID','tenure','Tenure Group']])

      customerID  tenure Tenure Group
0     7590-VHVEG       1    <12 tháng
1     5575-GNVDE      34  12-36 tháng
2     3668-QPYBK       2    <12 tháng
3     7795-CFOCW      45    >36 tháng
4     9237-HQITU       2    <12 tháng
...          ...     ...          ...
7038  6840-RESVB      24  12-36 tháng
7039  2234-XADUH      72    >36 tháng
7040  4801-JZAZL      11    <12 tháng
7041  8361-LTMKD       4    <12 tháng
7042  3186-AJIEK      66    >36 tháng

[7032 rows x 3 columns]


In [64]:
#Total Services (đếm số dịch vụ như PhoneService, MultipleLines, InternetService).
#Kiểm tra giá trị từng cột
print("PhoneService:", df_clean['PhoneService'].unique())
print("MultipleLines:", df_clean['MultipleLines'].unique())
print("InternetService:", df_clean['InternetService'].unique())

#Chuyển giá trị sang true => 1 => tính tổng
df_clean['Total Services'] = (
    (df_clean['PhoneService'] == 'Yes').astype(int) + 
    (df_clean['MultipleLines'] =='Yes').astype(int) +
    (df_clean['InternetService'].isin(['DSL','Fiber optic'])).astype(int)
    )
print(df_clean[['customerID', 'PhoneService', 'MultipleLines', 'InternetService', 'Total Services']].head(5))

PhoneService: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
   customerID PhoneService     MultipleLines InternetService  Total Services
0  7590-VHVEG           No  No phone service             DSL               1
1  5575-GNVDE          Yes                No             DSL               2
2  3668-QPYBK          Yes                No             DSL               2
3  7795-CFOCW           No  No phone service             DSL               1
4  9237-HQITU          Yes                No     Fiber optic               2


In [65]:
df_clean.to_csv('Telco_Customer_Churn_Cleaned.csv', index=False,encoding='utf-8-sig')